# LeetCode #332: Reconstruct Itinerary

https://leetcode.com/problems/reconstruct-itinerary/

## Comparison of Approaches
| Approach | Time | Space |
|---|---|---|
| Hierholzer's Algorithm ★ | O(E log E) | O(E) |
| DFS Backtracking | O(E!) | O(E) |

## Understanding the Methods
### Hierholzer's Algorithm (Optimal)
This problem is finding an Eulerian path in a directed graph. Sort adjacency lists lexicographically, then use Hierholzer's algorithm: greedily visit the smallest unvisited neighbor, and when stuck, add the current airport to the front of the result. This naturally produces the lexicographically smallest itinerary. The sorting dominates at O(E log E).

### DFS Backtracking
Try all orderings via backtracking, pruning non-lexicographic paths. Much slower in the worst case due to exponential branching.

## Solutions

### C#

In [ ]:
public class Solution {
    public IList<string> FindItinerary(IList<IList<string>> tickets) {
        var graph = new Dictionary<string, SortedDictionary<string, int>>();
        foreach (var t in tickets) {
            if (!graph.ContainsKey(t[0])) graph[t[0]] = new SortedDictionary<string, int>();
            var dest = graph[t[0]];
            dest[t[1]] = dest.GetValueOrDefault(t[1]) + 1;
        }

        var result = new LinkedList<string>();
        void Dfs(string airport) {
            if (graph.ContainsKey(airport)) {
                var dests = graph[airport];
                while (dests.Count > 0) {
                    var next = dests.First();
                    if (next.Value == 1) dests.Remove(next.Key);
                    else dests[next.Key]--;
                    Dfs(next.Key);
                }
            }
            result.AddFirst(airport);
        }
        Dfs("JFK");
        return result.ToList();
    }
}

### Python

In [ ]:
from collections import defaultdict

class Solution:
    def findItinerary(self, tickets: list[list[str]]) -> list[str]:
        graph = defaultdict(list)
        for src, dst in sorted(tickets, reverse=True):
            graph[src].append(dst)

        result = []
        def dfs(airport: str) -> None:
            while graph[airport]:
                dfs(graph[airport].pop())
            result.append(airport)

        dfs('JFK')
        return result[::-1]

### Go

In [ ]:
import "sort"

func findItinerary(tickets [][]string) []string {
    graph := map[string][]string{}
    for _, t := range tickets {
        graph[t[0]] = append(graph[t[0]], t[1])
    }
    for k := range graph {
        sort.Sort(sort.Reverse(sort.StringSlice(graph[k])))
    }

    var result []string
    var dfs func(string)
    dfs = func(airport string) {
        for len(graph[airport]) > 0 {
            next := graph[airport][len(graph[airport])-1]
            graph[airport] = graph[airport][:len(graph[airport])-1]
            dfs(next)
        }
        result = append(result, airport)
    }
    dfs("JFK")

    for i, j := 0, len(result)-1; i < j; i, j = i+1, j-1 {
        result[i], result[j] = result[j], result[i]
    }
    return result
}

### Rust

In [ ]:
use std::collections::HashMap;

impl Solution {
    pub fn find_itinerary(tickets: Vec<Vec<String>>) -> Vec<String> {
        let mut graph: HashMap<String, Vec<String>> = HashMap::new();
        for t in &tickets {
            graph.entry(t[0].clone()).or_default().push(t[1].clone());
        }
        for (_, dests) in graph.iter_mut() {
            dests.sort_unstable_by(|a, b| b.cmp(a));
        }

        let mut result = Vec::new();
        let mut stack = vec!["JFK".to_string()];
        while let Some(airport) = stack.last() {
            if graph.get(airport).map_or(true, |d| d.is_empty()) {
                result.push(stack.pop().unwrap());
            } else {
                let next = graph.get_mut(stack.last().unwrap()).unwrap().pop().unwrap();
                stack.push(next);
            }
        }
        result.reverse();
        result
    }
}

## Example Scenarios

### 1. Standard Itinerary
**Input:** `tickets = [["MUC","LHR"],["JFK","MUC"],["SFO","SJC"],["LHR","SFO"]]`  
Path: JFK -> MUC -> LHR -> SFO -> SJC. **Output:** `["JFK","MUC","LHR","SFO","SJC"]`

### 2. Lexicographic Choice
**Input:** `tickets = [["JFK","SFO"],["JFK","ATL"],["SFO","ATL"],["ATL","JFK"],["ATL","SFO"]]`  
Must pick lexicographically smallest at each step.

### 3. Single Ticket
**Input:** `tickets = [["JFK","LAX"]]`  
Direct flight. **Output:** `["JFK","LAX"]`

### 4. Round Trip
**Input:** `tickets = [["JFK","ATL"],["ATL","JFK"]]`  
JFK -> ATL -> JFK uses both tickets.

### 5. Multiple Tickets Same Route
**Input:** `tickets = [["JFK","ATL"],["JFK","ATL"],["ATL","JFK"]]`  
Duplicate tickets: JFK -> ATL -> JFK -> ATL.

![image](attachment:image.png)